<a id="top"></a>
# Paper 2+ — Diagnostik `ReadoutProbe`: Apakah `polyp_removal_status` & Kategori Lain Gagal karena *Perception* atau *Readout*?

Notebook ini **tidak melatih apa pun**. Ia menjawab satu pertanyaan yang menentukan apakah target **Gautam +3pp** dapat dicapai, sebelum GPU dipakai untuk Ortho-Unified:

> Ketika model menjawab salah, apakah informasinya **tidak ada di fitur visual** (*perception failure*) atau **ada tetapi gagal keluar sebagai teks** (*readout failure*)?

Metodenya mengikuti pola yang **sudah tervalidasi di Part 2** proyek ini: probe linier pada fitur beku menemukan `polyp_size` AUC $0{,}6998$ (kontrol acak $0{,}5001$) sementara generasi hanya $0{,}35$ — didiagnosis sebagai *readout failure*, dan *auxiliary head* kemudian benar menaikkannya $+15$pp.

---

### 📌 Mengapa diagnostik ini dijalankan lebih dulu

Plafon fusi SA-Bridge + GeoFiLM dihitung dari CSV judge resmi (`comprehensive_18cats_full_judge_qwen25_7b.csv`):

| | Qwen judge (ALL) |
| :--- | :--- |
| `M4_GeoFiLM` | 84,30 |
| `E_CLBRIDGE` | 84,56 |
| **Oracle fusi (max per kategori)** | **85,17** |
| `Gautam_Qwen2.5VL_7B` | 84,52 |
| **Selisih oracle $-$ Gautam** | **+0,65 pp** |
| Ambang target $+3$pp | **87,52** |

Artinya **fusi saja tidak cukup** — bahkan fusi sempurna hanya $+0{,}65$pp. Sisa $\approx 2{,}4$pp harus datang dari kategori yang masih punya ruang, dan ruang terbesar ada di sini:

| Kategori | $N$ | Judge ALL | Judge **L1** | Headroom (dibobot $N$) |
| :--- | :---: | :---: | :---: | :---: |
| **`polyp_removal_status`** | 2312 | 34,90 | **0,85** | **4,76 pp** |
| `abnormality_location` | 1901 | 68,28 | **12,27** | 1,91 pp |
| `landmark_location` | 1224 | 75,74 | 69,63 | 0,94 pp |
| `polyp_type` | 1959 | 85,96 | 87,35 | 0,87 pp |

`polyp_removal_status` pada L1 $= 0{,}85\%$ — praktis nol, padahal $N$-nya terbesar di dataset. **Probe inilah yang memutuskan apakah 4,76pp itu dapat dipanen.**

---

### 🧭 Navigasi Cepat Antar Bagian:
| Bagian | Deskripsi Singkat | Jalur Eksekusi |
| :--- | :--- | :--- |
| **[PART A — Persiapan](#part-a)** | Environment, path, modul, test set, & tabel patokan judge (`[A1]`–`[A6]`) | **Wajib** |
| **[PART B — Ekstraksi Fitur Beku](#part-b)** | Muat checkpoint, ekstrak DaViT $577 \times 768$ per gambar unik, cache `.npy` (`[B1]`–`[B2]`) | **Wajib** (~5–10 menit) |
| **[PART C — Label Kanonik](#part-c)** | Bangun label per kategori dari kolom `original`, tanpa daftar kata manual (`[C1]`) | **Wajib** |
| **[PART D — Probe Linier](#part-d)** | GroupKFold(`img_id`) + kontrol label teracak (`[D1]`–`[D2]`) | **Wajib** (~3 menit) |
| **[PART E — Putusan](#part-e)** | Bandingkan probe vs judge L1 $\rightarrow$ *readout* atau *perception* (`[E1]`) | **Wajib** |

### 📋 Catatan Metodologis (wajib dipertahankan di naskah)
1. **GroupKFold pada `img_id`**, bukan stratified — Kvasir-VQA-x1 punya 94,2% tumpang-tindih gambar antar partisi, sehingga split per-baris akan membocorkan gambar yang sama ke train dan val probe.
2. **Perbandingan hanya pada populasi atomik (complexity-1).** Probe membaca satu aspek per gambar, jadi pembandingnya wajib **judge L1**, bukan judge ALL. Kesalahan membandingkan probe (atomik) dengan judge ALL (campuran L2/L3 yang lebih mudah) menghasilkan "celah" palsu.
3. **Kontrol label teracak wajib dilaporkan.** Probe linier pada fitur 1536-dim dengan beberapa ribu sampel dapat menghafal; hanya selisih terhadap kontrol yang bermakna.

# PART A — PERSIAPAN
<a id="part-a"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top) | [Lompat ke PART B (Ekstraksi Fitur)](#part-b) | [Lompat ke PART E (Putusan)](#part-e)

# [A1] Environment — install paket

In [ ]:
# [A1] Environment — install paket & restart runtime sekali
!pip install -q transformers==4.49.0 peft "torchao>=0.16.0" scikit-learn
import transformers, peft, sklearn
print("transformers:", transformers.__version__, "| peft:", peft.__version__, "| sklearn:", sklearn.__version__)
print("\n>>> JIKA BARU INSTAL PAKET, RESTART RUNTIME SEKARANG, lalu lanjutkan mulai dari Cell [A2] <<<")

# [A2] Import & Mount Google Drive

In [ ]:
# [A2] Import pustaka & Mount Google Drive (otomatis dilewati jika berjalan di mesin lokal)
import os, sys, json, re, math, time, random, shutil, importlib
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", IS_COLAB, "| device:", dev,
      torch.cuda.get_device_name(0) if dev == "cuda" else "")

# [A3] Konfigurasi path & seed

In [ ]:
# [A3] Konfigurasi path, seed, & caching gambar ke disk lokal (/content/all_images di Colab)
if IS_COLAB:
    BASE  = Path("/content/drive/MyDrive/DOKTORAL/KvasirVQAX1")
    LOCAL = Path("/content/all_images")
else:
    BASE  = Path(r"g:/My Drive/DOKTORAL/KvasirVQAX1")
    LOCAL = BASE / "all_images"

ZIP_PATH = BASE / "all_images.zip"
t0_img = time.time()
if IS_COLAB and (not LOCAL.exists() or len(os.listdir(LOCAL)) < 6449):
    LOCAL.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        shutil.copy2(ZIP_PATH, "/content/all_images.zip")
        os.system("unzip -q -o /content/all_images.zip -d /content/all_images")
        os.remove("/content/all_images.zip")
        print(f"✓ Ekstrak gambar {time.time()-t0_img:.1f}s | total {len(os.listdir(LOCAL))}")
else:
    print(f"✓ Folder gambar siap: {LOCAL} ({len(os.listdir(LOCAL)) if LOCAL.exists() else 0} file)")

IMG_DIR   = LOCAL
CORE_DIR  = BASE / "GeoFiLM_Paper2"
JUDGE_DIR = CORE_DIR / "dual_judge_eval"
PAPER2    = BASE / "experiments/Paper2_GeoFiLM_r32"
BASE_ID   = "microsoft/Florence-2-base-ft"
OUTDIR    = CORE_DIR / "readout_probe"
OUTDIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTDIR / "feat_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

for nm, p in [("all_images", IMG_DIR), ("geofilmpp_core", CORE_DIR / "geofilmpp_core.py"),
              ("dual_judge_eval", JUDGE_DIR), ("Paper2_GeoFiLM_r32", PAPER2)]:
    print("OK  " if p.exists() else "MISS", p)
print("OK  OUTDIR ->", OUTDIR)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# [A4] Muat `geofilmpp_core.py` (parser label kanonik)

In [ ]:
# [A4] Muat geofilmpp_core.py — dipakai hanya untuk parser label kanonik (as_list / atoms), bukan arsitektur
assert (CORE_DIR / "geofilmpp_core.py").exists(), f"geofilmpp_core.py tidak ada di {CORE_DIR}"
sys.path.insert(0, str(CORE_DIR))
import geofilmpp_core as G
importlib.reload(G)
from geofilmpp_core import as_list, atoms
print("✓ parser label kanonik dimuat: as_list(), atoms()")

# [A5] Muat Test Set & pilih populasi atomik (complexity-1)

In [ ]:
# [A5] Muat Test Set (15.955 QA) & pilih populasi atomik complexity-1 yang sebanding dengan probe
from datasets import load_from_disk

te = load_from_disk(str(BASE / "test")).to_pandas()
te["_img"] = te["img_id"].map(lambda s: re.sub(r"\.(jpg|jpeg|png)$", "", str(s), flags=re.I))
print("Total test rows :", len(te))
print("Kolom           :", [c for c in te.columns][:10])

assert "complexity" in te.columns, "Kolom `complexity` tidak ada — probe L1 tidak bisa dibandingkan apple-to-apple."
L1 = te[te["complexity"] == 1].copy()
print(f"Populasi atomik (complexity=1): {len(L1)} baris | {L1._img.nunique()} gambar unik")
print("\nJumlah baris L1 per kategori:")
print(L1["question_class"].value_counts().to_string())

# [A6] Muat tabel patokan Judge (pembanding generatif)

In [ ]:
# [A6] Muat tabel patokan Judge dari CSV resmi — kolom L1_Judge adalah pembanding yang sah untuk probe
JUDGE_CSV = JUDGE_DIR / "comprehensive_18cats_full_judge_qwen25_7b.csv"
assert JUDGE_CSV.exists(), f"CSV judge tidak ditemukan: {JUDGE_CSV}"
jq = pd.read_csv(JUDGE_CSV)

MODEL_COL = "M4_GeoFiLM_r32"          # patokan generatif yang dibandingkan dengan probe
GEN = jq[["Category", "N_Aspects", f"{MODEL_COL}_L1_Judge", f"{MODEL_COL}_FULL_ALL",
          "Gautam_Qwen2.5VL_7B_L1_Judge", "Gautam_Qwen2.5VL_7B_FULL_ALL"]].copy()
GEN.columns = ["kategori", "N", "gen_L1", "gen_ALL", "gautam_L1", "gautam_ALL"]
GEN = GEN.set_index("kategori")

# Kategori yang diprobe: urut berdasarkan headroom terbobot N (ruang kosong ke 100%)
W = GEN["N"] / GEN["N"].sum()
GEN["headroom_pp"] = ((100 - GEN["gen_ALL"]) * W).round(2)
PROBE_CATS = GEN.sort_values("headroom_pp", ascending=False).head(6).index.tolist()

print(f"Patokan generatif = {MODEL_COL} (judge Qwen2.5-7B)\n")
print(GEN.loc[PROBE_CATS, ["N", "gen_L1", "gen_ALL", "gautam_L1", "gautam_ALL", "headroom_pp"]].to_string())
print(f"\nKategori yang akan diprobe: {PROBE_CATS}")

# PART B — EKSTRAKSI FITUR VISUAL BEKU
<a id="part-b"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top) | [Lompat ke PART D (Probe)](#part-d)
>
> Fitur diekstrak **per gambar unik**, bukan per baris QA, karena representasi visual tidak bergantung pada pertanyaan. Head `GeoFiLMModule` **tidak dipasang** (`head_holder = None`), sehingga yang diukur adalah isi representasi visual hasil adaptasi LoRA — bukan efek modulasi GeoFiLM. Ini membuat putusan *perception vs readout* bersih dari pengaruh modul yang sedang diuji.

# [B1] Pilih & muat checkpoint sumber fitur

In [ ]:
# [B1] Pilih & muat checkpoint sumber fitur (LoRA + image_projection dari run M4, tanpa head GeoFiLM)
from unittest.mock import patch
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports
import transformers.dynamic_module_utils as dmu
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from safetensors.torch import load_file

# "m4_ckpt" = fitur hasil adaptasi LoRA run M4 (default, paling relevan)
# "base"    = Florence-2-base-ft murni tanpa fine-tuning (kontrol tambahan)
FEAT_SOURCE = "m4_ckpt"
assert FEAT_SOURCE in ("m4_ckpt", "base")

orig_get_imports = dmu.get_imports
def _skip_flash(name, *a, **k):
    return None if "flash_attn" in str(name) else orig_get_imports(name, *a, **k)
def _skip_flash_proc(f):
    return [i for i in get_imports(f) if i != "flash_attn"]

with patch("transformers.dynamic_module_utils.get_imports", _skip_flash_proc):
    processor = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)

def _unwrap(m):
    seen = set()
    def walk(x, d=0):
        if d > 6 or id(x) in seen: return None
        seen.add(id(x))
        if "_encode_image" in type(x).__dict__: return x
        for nm in ("base_model", "model"):
            c = getattr(x, nm, None)
            if c is not None and c is not x:
                r = walk(c, d + 1)
                if r is not None: return r
        return None
    f = walk(m)
    assert f is not None, "could not locate the module defining _encode_image"
    return f

CKPT = None
if FEAT_SOURCE == "m4_ckpt":
    cands = sorted([p for p in PAPER2.glob("*M4_GeoFiLM*") if (p / "checkpoints/final").exists()])
    assert cands, f"Checkpoint M4 tidak ditemukan di {PAPER2}"
    CKPT = cands[-1] / "checkpoints/final"
    print("✓ Checkpoint M4 :", CKPT.parent.parent.name)

with patch("transformers.dynamic_module_utils.get_imports", _skip_flash):
    model = AutoModelForCausalLM.from_pretrained(BASE_ID, trust_remote_code=True,
                                                 torch_dtype=torch.float32)

if CKPT is not None:
    lora_json = json.loads((CKPT / "adapter_config.json").read_text())
    model = get_peft_model(model, LoraConfig(
        r=int(lora_json["r"]), lora_alpha=int(lora_json["lora_alpha"]),
        lora_dropout=lora_json.get("lora_dropout", 0.05),
        target_modules=list(lora_json["target_modules"]),
        bias=lora_json.get("bias", "none"), task_type=lora_json.get("task_type")))
    set_peft_model_state_dict(model, load_file(str(CKPT / "adapter_model.safetensors")))
    core = _unwrap(model)
    if (CKPT / "image_proj.pt").exists():
        ip = torch.load(CKPT / "image_proj.pt", map_location="cpu")
        core.image_projection.data.copy_(ip["image_projection"].to(core.image_projection.dtype))
        core.image_proj_norm.load_state_dict(ip["image_proj_norm"])
        print("✓ image_projection & image_proj_norm dipulihkan")
    print(f"✓ LoRA r={lora_json['r']} dimuat dari checkpoint")
else:
    core = _unwrap(model)
    print("✓ Memakai Florence-2-base-ft murni (tanpa fine-tuning)")

model = model.to(dev).eval()
for p in model.parameters():
    p.requires_grad_(False)
print("✓ Model BEKU. FEAT_SOURCE =", FEAT_SOURCE)

# [B2] Ekstrak fitur DaViT per gambar unik & cache ke disk

In [ ]:
# [B2] Ekstrak fitur DaViT (577 x 768) per gambar unik, lalu ringkas jadi [CLS ; mean-patch] dan cache .npy
from PIL import Image
from tqdm.auto import tqdm

FEAT_PATH = CACHE_DIR / f"feats_{FEAT_SOURCE}.npz"
imgs_needed = sorted(L1["_img"].unique().tolist())
print(f"Gambar unik yang dibutuhkan: {len(imgs_needed)}")

if FEAT_PATH.exists():
    z = np.load(FEAT_PATH, allow_pickle=True)
    FEAT = {k: v for k, v in zip(z["ids"], z["X"])}
    print(f"✓ Cache ditemukan ({len(FEAT)} gambar) -> {FEAT_PATH.name}")
    missing = [i for i in imgs_needed if i not in FEAT]
    print(f"  gambar belum ter-cache: {len(missing)}")
else:
    FEAT, missing = {}, imgs_needed

if missing:
    BS = 32
    with torch.no_grad():
        for s in tqdm(range(0, len(missing), BS), desc="ekstrak fitur"):
            chunk = missing[s:s + BS]
            pil = [Image.open(IMG_DIR / f"{i}.jpg").convert("RGB") for i in chunk]
            px = processor(images=pil, text=["<VQA> x"] * len(pil),
                           return_tensors="pt", padding=True)["pixel_values"].to(dev)
            with torch.amp.autocast("cuda", enabled=(dev == "cuda")):
                f = core._encode_image(px)              # (B, 577, 768)
            f = f.float()
            # Ringkasan: token global (CLS) + rata-rata 576 patch -> 1536-dim
            summ = torch.cat([f[:, 0, :], f[:, 1:, :].mean(1)], dim=-1).cpu().numpy()
            for i, v in zip(chunk, summ):
                FEAT[i] = v.astype(np.float32)
    ids = np.array(list(FEAT.keys()))
    X   = np.stack([FEAT[i] for i in ids])
    np.savez_compressed(FEAT_PATH, ids=ids, X=X)
    print(f"✓ Cache disimpan: {FEAT_PATH} | shape {X.shape}")

D_FEAT = len(next(iter(FEAT.values())))
print(f"✓ Siap: {len(FEAT)} gambar, dimensi fitur {D_FEAT} ([CLS 768 ; mean-patch 768])")

# PART C — LABEL KANONIK DARI KOLOM `original`
<a id="part-c"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top) | [Lompat ke PART D (Probe)](#part-d)
>
> Label **tidak ditulis tangan**: untuk setiap kategori, himpunan label diturunkan dari kolom `original` milik dataset (ambang $\ge 20$ kemunculan membuang *typo*/*outlier*). Kategori yang satu barisnya dapat memuat beberapa atom (mis. lokasi dengan beberapa sel) otomatis diperlakukan sebagai **multi-label**; sisanya **multi-kelas**.

# [C1] Bangun label kanonik & tentukan mode probe per kategori

In [ ]:
# [C1] Bangun label kanonik per kategori dari kolom `original` & tentukan mode probe (multiclass / multilabel)
MIN_COUNT = 20

def build_labels(df_cat):
    """Kembalikan (rows, vocab, mode). rows = list dict{img, atoms}. mode: 'multiclass' | 'multilabel'."""
    cnt = Counter()
    recs = []
    for _, r in df_cat.iterrows():
        aset = []
        for sub in as_list(r.get("original")):
            for a in atoms(sub):
                if a and a not in aset:
                    aset.append(a)
        if not aset:
            continue
        for a in aset:
            cnt[a] += 1
        recs.append({"img": r["_img"], "atoms": aset})
    vocab = sorted([a for a, n in cnt.items() if n >= MIN_COUNT])
    recs  = [{"img": r["img"], "atoms": [a for a in r["atoms"] if a in vocab]} for r in recs]
    recs  = [r for r in recs if r["atoms"]]
    multi = float(np.mean([len(r["atoms"]) > 1 for r in recs])) if recs else 0.0
    mode  = "multilabel" if multi > 0.05 else "multiclass"
    return recs, vocab, mode, multi

LAB = {}
rows_sum = []
for c in PROBE_CATS:
    sub = L1[L1["question_class"] == c]
    recs, vocab, mode, multi = build_labels(sub)
    LAB[c] = {"recs": recs, "vocab": vocab, "mode": mode}
    maj = 0.0
    if recs:
        if mode == "multiclass":
            lab = [r["atoms"][0] for r in recs]
            maj = 100 * Counter(lab).most_common(1)[0][1] / len(lab)
        else:  # baseline multilabel = selalu prediksi atom paling sering
            maj = 100 * np.mean([vocab and (Counter([a for r in recs for a in r["atoms"]])
                                            .most_common(1)[0][0] in r["atoms"]) for r in recs])
    rows_sum.append({"kategori": c, "baris_L1": len(sub), "terpakai": len(recs),
                     "n_label": len(vocab), "mode": mode,
                     "rata2_atom": round(float(np.mean([len(r["atoms"]) for r in recs])), 2) if recs else 0,
                     "majority_%": round(maj, 2)})

df_lab = pd.DataFrame(rows_sum)
print(df_lab.to_string(index=False))
print("\nContoh label per kategori:")
for c in PROBE_CATS:
    v = LAB[c]["vocab"]
    print(f"  {c:24s} [{LAB[c]['mode']:10s}] {len(v):3d} label -> {v[:6]}{' ...' if len(v) > 6 else ''}")

# PART D — PROBE LINIER DENGAN GroupKFold & KONTROL LABEL TERACAK
<a id="part-d"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top) | [Lompat ke PART E (Putusan)](#part-e)
>
> **GroupKFold pada `img_id`** wajib: Kvasir-VQA-x1 memiliki 94,2% tumpang-tindih gambar antar partisi, sehingga split acak per-baris akan menempatkan gambar yang sama di train dan val probe dan melaporkan AUC yang terlalu optimistis.
>
> **Kontrol label teracak** dijalankan untuk setiap kategori. Hanya selisih `AUC_asli − AUC_teracak` yang dapat ditafsirkan; AUC absolut pada fitur 1536-dim dengan beberapa ribu sampel dapat terangkat oleh kapasitas probe itu sendiri.

# [D1] Fungsi probe linier (GroupKFold + kontrol teracak)

In [ ]:
# [D1] Fungsi probe linier (GroupKFold pada img_id + kontrol label teracak)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")

N_SPLITS = 5
MIN_POS  = 20          # atom dengan positif < MIN_POS dilewati (AUC tidak stabil)

def _xy(recs, vocab, mode):
    X = np.stack([FEAT[r["img"]] for r in recs])
    g = np.array([r["img"] for r in recs])
    if mode == "multiclass":
        v2i = {a: i for i, a in enumerate(vocab)}
        y = np.array([v2i[r["atoms"][0]] for r in recs])
    else:
        y = np.zeros((len(recs), len(vocab)), dtype=np.int8)
        v2i = {a: i for i, a in enumerate(vocab)}
        for i, r in enumerate(recs):
            for a in r["atoms"]:
                y[i, v2i[a]] = 1
    return X, y, g

def run_probe(recs, vocab, mode, shuffle=False, seed=0, C=1.0):
    """Kembalikan dict metrik. multiclass -> acc + macro-AUC; multilabel -> macro-AUC per atom.

    C = kekuatan regularisasi. Dijalankan pada dua nilai sebagai uji kestabilan: putusan
    tidak boleh bergantung pada pilihan regularisasi. Catatan hasil uji sintetis: C kecil
    TIDAK menyaring sinyal beramplitudo rendah (regularisasi menekan hafalan derau, bukan
    arah linier yang konsisten), jadi pembeda sesungguhnya antara 'informasi ada' dan
    'tidak ada' adalah KONTROL LABEL TERACAK, bukan sweep C ini.
    """
    X, y, g = _xy(recs, vocab, mode)
    rng = np.random.RandomState(seed)
    if shuffle:
        idx = rng.permutation(len(y))
        y = y[idx]
    gkf = GroupKFold(n_splits=min(N_SPLITS, len(np.unique(g))))

    if mode == "multiclass":
        accs, aucs = [], []
        for tr_i, va_i in gkf.split(X, y, groups=g):
            if len(np.unique(y[tr_i])) < 2:
                continue
            sc = StandardScaler().fit(X[tr_i])
            # Tanpa argumen multi_class: sudah dihapus di scikit-learn >= 1.7,
            # dan perilaku multinomial kini menjadi default — notebook ini harus jalan di sklearn lama maupun baru.
            clf = LogisticRegression(max_iter=2000, C=C)
            clf.fit(sc.transform(X[tr_i]), y[tr_i])
            pr = clf.predict(sc.transform(X[va_i]))
            accs.append(accuracy_score(y[va_i], pr))
            try:
                pp = clf.predict_proba(sc.transform(X[va_i]))
                present = np.unique(y[va_i])
                if len(present) > 1:
                    cols = [list(clf.classes_).index(c) for c in present if c in clf.classes_]
                    aucs.append(roc_auc_score(y[va_i], pp[:, cols], multi_class="ovr",
                                              average="macro", labels=present))
            except Exception:
                pass
        return {"acc": float(np.mean(accs)) if accs else np.nan,
                "auc": float(np.mean(aucs)) if aucs else np.nan,
                "n": len(y), "n_label": len(vocab)}

    # multilabel: macro-AUC rata-rata antar atom yang punya cukup positif
    per_atom = []
    for j in range(y.shape[1]):
        if y[:, j].sum() < MIN_POS or y[:, j].sum() > len(y) - MIN_POS:
            continue
        a_fold = []
        for tr_i, va_i in gkf.split(X, y[:, j], groups=g):
            if len(np.unique(y[tr_i, j])) < 2 or len(np.unique(y[va_i, j])) < 2:
                continue
            sc = StandardScaler().fit(X[tr_i])
            clf = LogisticRegression(max_iter=2000, C=C)
            clf.fit(sc.transform(X[tr_i]), y[tr_i, j])
            a_fold.append(roc_auc_score(y[va_i, j],
                                        clf.predict_proba(sc.transform(X[va_i]))[:, 1]))
        if a_fold:
            per_atom.append(float(np.mean(a_fold)))
    return {"acc": np.nan, "auc": float(np.mean(per_atom)) if per_atom else np.nan,
            "n": len(y), "n_label": len(vocab), "n_atom_dipakai": len(per_atom)}

print("✓ Fungsi probe siap | GroupKFold splits =", N_SPLITS, "| MIN_POS =", MIN_POS)

# [D2] Jalankan probe untuk semua kategori

In [ ]:
# [D2] Jalankan probe untuk semua kategori (asli + kontrol teracak)
res = []
for c in PROBE_CATS:
    d = LAB[c]
    if len(d["recs"]) < 50 or len(d["vocab"]) < 2:
        print(f"  {c:24s} SKIP (recs={len(d['recs'])}, labels={len(d['vocab'])})")
        continue
    t0 = time.time()
    real = run_probe(d["recs"], d["vocab"], d["mode"], shuffle=False, seed=SEED, C=1.0)
    ctrl = run_probe(d["recs"], d["vocab"], d["mode"], shuffle=True,  seed=SEED, C=1.0)
    # Uji kestabilan: putusan tidak boleh berubah karena pilihan regularisasi
    strg = run_probe(d["recs"], d["vocab"], d["mode"], shuffle=False, seed=SEED, C=0.01)
    row = {"kategori": c, "mode": d["mode"], "n": real["n"], "n_label": real["n_label"],
           "probe_acc": real["acc"], "probe_auc": real["auc"],
           "ctrl_acc": ctrl["acc"], "ctrl_auc": ctrl["auc"],
           "probe_auc_C001": strg["auc"],
           "d_auc": (real["auc"] - ctrl["auc"]) if not (np.isnan(real["auc"]) or np.isnan(ctrl["auc"])) else np.nan,
           "d_auc_C001": (strg["auc"] - ctrl["auc"]) if not (np.isnan(strg["auc"]) or np.isnan(ctrl["auc"])) else np.nan}
    res.append(row)
    print(f"  {c:24s} [{d['mode']:10s}] n={real['n']:5d} | "
          f"AUC {real['auc']:.4f} (kontrol {ctrl['auc']:.4f}, Δ {row['d_auc']:+.4f}) "
          f"| C=0.01: {strg['auc']:.4f} (Δ {row['d_auc_C001']:+.4f}) | {time.time()-t0:.0f}s")

df_probe = pd.DataFrame(res)
df_probe.to_csv(OUTDIR / f"probe_results_{FEAT_SOURCE}.csv", index=False)
print(f"\n✓ disimpan -> {OUTDIR / f'probe_results_{FEAT_SOURCE}.csv'}")
display(df_probe.round(4))

# PART E — PUTUSAN: *READOUT* ATAU *PERCEPTION*?
<a id="part-e"></a>
> [⬆️ Kembali ke Navigasi Cepat](#top)
>
> Aturan putusan, mengikuti preseden `polyp_size` di Part 2 (probe AUC $0{,}6998$ vs kontrol $0{,}5001$, generasi $0{,}35$ $\rightarrow$ *readout*, dan *auxiliary head* kemudian benar menaikkannya $+15$pp):
>
> | Kondisi | Putusan | Tindakan yang dibenarkan |
> | :--- | :--- | :--- |
> | $\Delta$AUC $\ge 0{,}15$ **dan** generasi L1 rendah | **READOUT** — informasi ada, gagal keluar sebagai teks | **Slot kanonik** (sisi data, 0 parameter). Panen besar, murah. |
> | $\Delta$AUC $0{,}05$–$0{,}15$ | **CAMPURAN** | Slot dulu; *auxiliary head* hanya jika slot kurang. |
> | $\Delta$AUC $< 0{,}05$ | **PERCEPTION** — informasi tidak ada di fitur | Slot **tidak akan** menolong. Perlu *auxiliary head* / kapasitas tambahan, dan perkiraan $+3$pp harus direvisi turun. |
>
> $\Delta$AUC diukur terhadap **kontrol label teracak**, dan diambil nilai paling konservatif dari dua kekuatan regularisasi ($C = 1{,}0$ dan $C = 0{,}01$).
>
> Perbandingan memakai **judge L1**, bukan judge ALL, karena probe membaca satu aspek per gambar.
>
> **Dua batas tafsir yang wajib disebut di naskah.** (1) Probe linier menunjukkan informasi itu **ada secara linier** pada fitur; itu **batas atas** keterbacaan, bukan jaminan decoder mampu memakainya. Pembenaran untuk bertindak atasnya adalah preseden `polyp_size` di Part 2 — probe $0{,}6998$ lalu *auxiliary head* benar memberi $+15$pp. (2) Uji sintetis pada pipeline ini memperlihatkan probe 1536-dim dapat memulihkan arah linier beramplitudo sangat kecil; karena itu AUC absolut tidak dipakai sebagai klaim, hanya selisihnya terhadap kontrol teracak.

# [E1] Tabel putusan & estimasi dampak pada Overall

In [ ]:
# [E1] Tabel putusan (probe vs judge L1) & estimasi dampak terbobot N pada Overall
if df_probe.empty:
    raise RuntimeError("df_probe kosong — periksa [D2].")

v = df_probe.set_index("kategori").join(GEN[["N", "gen_L1", "gen_ALL", "gautam_L1", "gautam_ALL"]])
v["probe_auc"] = v["probe_auc"].astype(float)

def verdict(r):
    # Dasar putusan = Δ terhadap kontrol teracak, diambil yang paling konservatif dari dua
    # nilai regularisasi. Jika keduanya berbeda jauh, hasilnya tidak stabil -> jangan diklaim.
    cand = [x for x in (r.get("d_auc"), r.get("d_auc_C001")) if x is not None and not np.isnan(x)]
    d = min(cand) if cand else np.nan
    if np.isnan(d):
        return "TIDAK TERUKUR"
    if d >= 0.15:
        return "READOUT"
    if d >= 0.05:
        return "CAMPURAN"
    return "PERCEPTION"

v["putusan"] = v.apply(verdict, axis=1)
Wn = v["N"] / GEN["N"].sum()
# Estimasi optimistis: kategori READOUT naik ke tingkat kategori yang readout-nya sudah sehat (90% dari AUC-nya)
v["est_naik_pp"] = np.where(v["putusan"].eq("READOUT"),
                            ((100 * v["probe_auc_C001"] * 0.90) - v["gen_ALL"]).clip(lower=0) * Wn, 0.0)

show = v[["N", "mode", "n", "probe_auc", "ctrl_auc", "d_auc", "probe_auc_C001", "d_auc_C001",
          "gen_L1", "gen_ALL", "gautam_ALL", "putusan", "est_naik_pp"]]
print("=" * 108)
print(" TABEL PUTUSAN — probe (fitur beku) vs generatif (judge Qwen2.5-7B)")
print("=" * 108)
display(show.round(4))

tot = float(v["est_naik_pp"].sum())
ORACLE_FUSI, GAUTAM_ALL = 85.17, 84.52      # dihitung di header dari CSV judge resmi
print(f"\nEstimasi kenaikan dari kategori READOUT (terbobot N) : {tot:+.2f} pp")
print(f"Oracle fusi SA-Bridge + GeoFiLM                       : {ORACLE_FUSI:.2f}  ({ORACLE_FUSI-GAUTAM_ALL:+.2f} vs Gautam)")
print(f"Proyeksi dengan slot kanonik pada kategori READOUT    : {ORACLE_FUSI+tot:.2f}  ({ORACLE_FUSI+tot-GAUTAM_ALL:+.2f} vs Gautam)")
print(f"Ambang target                                         : {GAUTAM_ALL+3:.2f}  (Gautam +3pp)")
print("\n" + ("✓ TARGET +3pp MASIH DI JALUR — lanjutkan ke slot kanonik untuk kategori READOUT."
               if ORACLE_FUSI + tot >= GAUTAM_ALL + 3 else
               "✗ TARGET +3pp BELUM TERCAPAI dari jalur readout saja.\n"
               "  Opsi: (a) auxiliary head untuk kategori PERCEPTION (menambah kapasitas),\n"
               "        (b) revisi target ke angka yang didukung bukti (mis. paritas statistik + unggul per-kategori)."))

v.round(4).to_csv(OUTDIR / f"verdict_{FEAT_SOURCE}.csv")
print(f"\n✓ disimpan -> {OUTDIR / f'verdict_{FEAT_SOURCE}.csv'}")

---
## Cara Membaca Hasil & Langkah Lanjutan

**Jika `polyp_removal_status` keluar `READOUT`** (ΔAUC $\ge 0{,}15$ dengan generasi L1 $0{,}85\%$) — ini skenario terbaik. Kategori dengan $N = 2312$ dan *headroom* $4{,}76$pp dapat dipanen lewat **slot kanonik sisi data, nol parameter**. Tambahkan `polyp_removal_status` ke daftar kategori ber-slot di `geofilmpp_core.py` (fungsi `build_loc_slot` digeneralisasi menjadi `build_cat_slot`), lalu latih ulang `U2_slot`.

**Jika keluar `PERCEPTION`** — slot tidak akan menolong. Yang dibenarkan adalah *auxiliary head* khusus seperti CORN untuk ukuran: head diskriminatif pada `f_fg` dengan bobot $\lambda$ kecil, dilatih bersama VQA, dibuang atau dipertahankan saat inferensi. Ini penambahan kapasitas yang tepat sasaran — tetapi perkiraan $+3$pp harus direvisi turun sampai ada bukti.

**Catatan jujur tentang target.** Oracle fusi SA-Bridge + GeoFiLM hanya $+0{,}65$pp terhadap Gautam. Angka $+3$pp **tidak** dapat dicapai oleh fusi; ia bergantung sepenuhnya pada apakah `polyp_removal_status` (dan kategori lokasi) ternyata *readout-limited*. Notebook ini memberi jawabannya dalam ~15 menit, sebelum ~9 jam GPU dipakai untuk arsitektur yang mungkin menyerang masalah yang salah.

**Konsistensi angka yang perlu dibereskan di naskah.** CSV judge resmi (`comprehensive_18cats_full_judge_qwen25_7b.csv`) memberi `polyp_removal_status` M4 $= 34{,}90$ dan Gautam $= 27{,}77$ (ALL), sedangkan tabel di `RINGKASAN_PIPELINE_DAN_ABLASI.md` menulis $49{,}4$ dan $38{,}1$. Selisih $\approx 14$pp pada metrik yang sama. CSV adalah keluaran langsung pipeline judge, jadi itu yang dipakai di notebook ini — tetapi kedua dokumen harus disatukan sebelum submit.